# FFT Causal Conv1D Backward

This notebook verifies input and filter gradients for the medium and long FFT paths exposed by `cudnn.ops.fft_causal_conv1d`.

## Prerequisites

An NVIDIA GPU, cuDNN 9.26.0 or newer, and a cuDNN Frontend Python binding built against cuDNN 9.26.0 or newer are required.

In [1]:
import math

import cudnn
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "This sample requires a CUDA device"
assert cudnn.backend_version() >= 92600, "FFT causal conv1d requires cuDNN 9.26.0 or newer"

In [2]:
def fft_causal_conv1d_reference(x, weight):
    dim, kernel_size = weight.shape
    x_padded = F.pad(x, (kernel_size - 1, 0))
    return F.conv1d(x_padded, weight.flip(-1).unsqueeze(1), groups=dim)

The long backward call consumes opaque frequency-domain reserve space saved by its matching forward call. The Python autograd registration manages that reserve-space lifetime automatically.

In [3]:
torch.manual_seed(42)
configs = [
    {"name": "medium-padded", "batch": 2, "dim": 4, "seq_len": 750, "kernel_size": 192, "dtype": torch.float32, "atol": 2e-6, "rtol": 2e-6},
    {"name": "long", "batch": 1, "dim": 1, "seq_len": 8192, "kernel_size": 8192, "dtype": torch.float64, "atol": 5e-11, "rtol": 5e-11},
]

for config in configs:
    x_data = 0.1 * torch.randn(config["batch"], config["dim"], config["seq_len"], device="cuda", dtype=config["dtype"])
    weight_data = torch.randn(config["dim"], config["kernel_size"], device="cuda", dtype=config["dtype"]) / math.sqrt(config["kernel_size"])
    grad_out = 0.1 * torch.randn_like(x_data)

    x = x_data.detach().requires_grad_(True)
    weight = weight_data.detach().requires_grad_(True)
    cudnn.ops.fft_causal_conv1d(x, weight).backward(grad_out)

    x_ref = x_data.double().detach().requires_grad_(True)
    weight_ref = weight_data.double().detach().requires_grad_(True)
    fft_causal_conv1d_reference(x_ref, weight_ref).backward(grad_out.double())

    dx_max_abs = (x.grad.double() - x_ref.grad).abs().max().item()
    dw_max_abs = (weight.grad.double() - weight_ref.grad).abs().max().item()
    print(f'{config["name"]}: dx_max_abs={dx_max_abs:.6e}, dw_max_abs={dw_max_abs:.6e}')
    torch.testing.assert_close(x.grad, x_ref.grad.to(config["dtype"]), atol=config["atol"], rtol=config["rtol"])
    torch.testing.assert_close(weight.grad, weight_ref.grad.to(config["dtype"]), atol=config["atol"], rtol=config["rtol"])

medium-padded: dx_max_abs=9.984049e-08, dw_max_abs=3.097257e-07


long: dx_max_abs=1.804112e-15, dw_max_abs=1.132427e-14
